# Part 2a - Baseline Price Predictor

> Case study part: Part 2 of 3 (first of two notebooks)
> Required for: All students (3-credit and 4-credit)
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

In Part 1 we described the campus-town markets. In Part 2 we want to
predict the price of any single-family home in QCP's footprint from its
observable features - the kind of automated valuation model (AVM) that
underpins Zillow's _Zestimate_ and Redfin's _Estimate_.

The target is `price`, which in this extract is the most recent sold price
(see the note in Part 1), not a current asking price. So the question here is:
from a home's size, age, and location alone, how close can we get to what it
last sold for?

This notebook builds the baseline model. It uses only structured tabular
features (location, beds, baths, square footage, lot size, age, parking, and
the anchoring `campus`). Part 2b will add NLP-derived features from the
listing description and test whether they improve the model.

By the end of this notebook you should be able to answer:

1. What does a defensible baseline price-prediction pipeline look like?
2. How well does it perform on a held-out sample (MAE, RMSE, R², MAPE)?
3. Where does it work well and where does it fail?

## ⚙️ Setup


In [1]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 60)

## 📥 Load the Cleaned Dataset from Part 1

If you have not run Part 1, do so first - it produces `properties_clean.parquet`.


In [2]:
DATA_DIR = "data"

df = pd.read_parquet(f"{DATA_DIR}/properties_clean.parquet")
print(f"{len(df):,} cleaned listings loaded")
df.head(3)


35,273 cleaned listings loaded


,property_id,date_posted,days_on_zillow,price,last_sold_price,description,beds,baths,year_built,latitude,longitude,lot_size,living_area,address_street,address_city,address_state,address_zipcode,address_county,address_country,address_raw,county_fips,reso_is_new_construction,reso_architectural_style,reso_hoa_fee,reso_parking_capacity,reso_elementary_school,reso_middle_or_junior_school,reso_high_school,price_per_sqft,age_years,market,campus,desc_word_count
0,55417927,2025-06-19,245.0,350000,350000,"Charming 2-bed, 1-bath home in Madison?s desir...",2,1.0,1949.0,43.095300,-89.327545,6098.0,747.0,105 Harding Street,Madison,WI,53714,Dane County,USA,105 Harding Street,55025.0,0.0,Bungalow,NaN,1,0.0,Whitehorse,Lafollette,468.540830,77.0,"Madison, WI",University of Wisconsin-Madison,98
1,55411151,2025-06-23,235.0,615000,615000,One of a kind mid-century modern dream in the ...,4,3.0,1948.0,43.041683,-89.475716,23522.0,1699.0,1118 WOODLAND Way,Madison,WI,53711,Dane County,USA,1118 WOODLAND Way,55025.0,0.0,Raised Ranch,NaN,2,1.0,Toki,Memorial,361.977634,78.0,"Madison, WI",University of Wisconsin-Madison,124
2,383203051,2025-05-08,288.0,335000,335000,Charming Ranch Home in Burr Oaks ? Move-In Rea...,3,2.0,1949.0,43.043194,-89.396610,6098.0,1436.0,849 Dane Street,Madison,WI,53713,Dane County,USA,849 Dane Street,55025.0,0.0,Ranch,NaN,0,0.0,Call School District,Call School District,233.286908,77.0,"Madison, WI",University of Wisconsin-Madison,126


## 🧱 Choose the Target and Features

Our target is `price`, the most recent sold price. We model `log(price)`
instead of raw price because real-estate prices are right-skewed and a log
target makes the model's percentage error roughly constant across price
bands - which matches how investors actually think ("within 10%" is more
useful than "within $50,000").

Define the target and the two feature lists (numeric and categorical).


In [3]:
TARGET = "price"

NUMERIC_FEATURES = [
    "beds",
    "baths",
    "living_area",
    "lot_size",
    "year_built",
    "reso_parking_capacity",
    "latitude",
    "longitude",
]
CATEGORICAL_FEATURES = [
    "address_state",
    "address_city",
    "address_zipcode",
    "campus",
    "reso_architectural_style",
]


Build the modeling frame and the log-transformed target.


In [4]:
model_df = df.dropna(subset=[TARGET]).copy()
model_df["log_price"] = np.log1p(model_df[TARGET])

X = model_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
# Zip code is numeric in the source but should be treated as a category.
X["address_zipcode"] = X["address_zipcode"].astype(str)
y_log = model_df["log_price"]
y_raw = model_df[TARGET]

X.head(3)


,beds,baths,living_area,lot_size,year_built,reso_parking_capacity,latitude,longitude,address_state,address_city,address_zipcode,campus,reso_architectural_style
0,2,1.0,747.0,6098.0,1949.0,1,43.095300,-89.327545,WI,Madison,53714,University of Wisconsin-Madison,Bungalow
1,4,3.0,1699.0,23522.0,1948.0,2,43.041683,-89.475716,WI,Madison,53711,University of Wisconsin-Madison,Raised Ranch
2,3,2.0,1436.0,6098.0,1949.0,0,43.043194,-89.396610,WI,Madison,53713,University of Wisconsin-Madison,Ranch


## ✂️ Train / Test Split

We hold out 20% of listings as a test set. We stratify by state so every state
appears in both splits in roughly the same proportion (otherwise a small
market like NC could end up entirely in one split and skew the metrics).


In [5]:
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X,
    y_log,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=model_df["address_state"],
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")


Train: 28,218   Test: 7,055


## 🔧 Build a Reproducible Pipeline

We bundle preprocessing and the model into a single `Pipeline` so the full
training step is one `.fit()` call and the inference step is one `.predict()`
call. This is the same pattern used in the _Australian Rental Market_ case.


Numeric and categorical preprocessors.


In [6]:
numeric_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, NUMERIC_FEATURES),
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)


Wrap it together with an XGBoost regressor.


In [7]:
baseline_model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "regressor",
            XGBRegressor(
                n_estimators=600,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=42,
                n_jobs=-1,
                tree_method="hist",
            ),
        ),
    ]
)


:::{note} Why XGBoost?

Gradient-boosted trees are the workhorse model for tabular real-estate data.
They handle non-linear relationships (price-per-sqft is not linear in size),
they cope gracefully with missing values after imputation, and they need
almost no scaling. We pick `XGBoost` because it is the de-facto industry
standard for this kind of tabular regression, trains quickly via the
`hist` tree method, and uses all available CPU cores out of the box.
:::


In [8]:
baseline_model.fit(X_train, y_train_log)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['beds','baths','living_area',...,'address_zipcode','campus', 'reso_architectural_style']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Th

## 📏 Evaluate on the Held-out Test Set

The model predicts `log(price)`, so we exponentiate before computing
business-friendly error metrics in dollars.


In [9]:
def evaluate(model, X_eval, y_eval_raw, label=""):
    pred_log = model.predict(X_eval)
    pred = np.expm1(pred_log)
    mae = mean_absolute_error(y_eval_raw, pred)
    rmse = np.sqrt(mean_squared_error(y_eval_raw, pred))
    r2 = r2_score(y_eval_raw, pred)
    mape = np.mean(np.abs((y_eval_raw - pred) / y_eval_raw)) * 100
    print(f"=== {label} ===")
    print(f"  MAE   ${mae:>12,.0f}")
    print(f"  RMSE  ${rmse:>12,.0f}")
    print(f"  R²     {r2:>12.3f}")
    print(f"  MAPE   {mape:>11.2f}%")
    return pred


baseline_pred_test = evaluate(baseline_model, X_test, y_test_raw, "Baseline - Test")


=== Baseline - Test ===
  MAE   $      73,864
  RMSE  $     148,556
  R²            0.802
  MAPE         17.08%


:::{tip} Talking the metrics in business language

- MAE: "On average, the model is off by \$X."
- RMSE: same idea, but penalizes a single very wrong prediction more
  than MAE does. Always >= MAE.
- R²: "The model explains X% of the variation in price."
- MAPE: "On average, the model is off by X%." This is the metric the
  Investment Committee will actually quote.

:::

## 🔍 Residual Analysis - Where Does the Model Fail?

An overall MAPE of about 17% could still hide a market where the model is
systematically wrong. We break the error down by market and price band.

Build the diagnostic frame and aggregate errors by market.


In [10]:
diag = pd.DataFrame(
    {
        "actual": y_test_raw.values,
        "predicted": baseline_pred_test,
        "market": model_df.loc[X_test.index, "market"].values,
        "campus": model_df.loc[X_test.index, "campus"].values,
    }
)
diag["abs_error"] = (diag["actual"] - diag["predicted"]).abs()
diag["pct_error"] = (diag["actual"] - diag["predicted"]) / diag["actual"] * 100

by_market = (
    diag.groupby("market", as_index=False)
    .agg(
        n=("actual", "count"),
        median_abs_error=("abs_error", "median"),
        median_pct_error=("pct_error", lambda s: s.abs().median()),
    )
    .sort_values("median_pct_error")
)
by_market


,market,n,median_abs_error,median_pct_error
12,"Savoy, IL",55,25008.437500,7.279410
15,"West Lafayette, IN",324,29246.828125,7.806014
6,"Davis, CA",190,67025.281250,7.886427
9,"Lafayette, IN",612,21008.578125,8.177991
10,"Madison, WI",1025,39465.812500,9.137102
4,"Chapel Hill, NC",547,72870.750000,9.725747
7,"Gainesville, FL",1125,33087.218750,9.748029
11,"Middleton, WI",124,64569.156250,10.127949
8,"Iowa City, IA",388,38031.031250,10.424578
13,"State College, PA",271,48854.125000,10.837977


In [11]:
fig = px.bar(
    by_market.sort_values("median_pct_error"),
    x="median_pct_error",
    y="market",
    orientation="h",
    title="Baseline model - median |% error| by market",
    labels={"median_pct_error": "Median |% error|", "market": ""},
    color="median_pct_error",
    color_continuous_scale="Reds",
    height=600,
)
fig.show()


The median error per market (about 7% to 12%) is well below the overall MAPE.
MAPE is a mean, so a minority of large misses pulls it up, while the typical
listing is predicted more closely. Quote both when you present the model.

Bucket residuals by price band to see where the model is biased.


In [12]:
diag["price_band"] = pd.cut(
    diag["actual"],
    bins=[0, 200_000, 350_000, 500_000, 750_000, 1_000_000, np.inf],
    labels=["<$200K", "$200-350K", "$350-500K", "$500-750K", "$750K-1M", ">$1M"],
)

fig = px.box(
    diag,
    x="price_band",
    y="pct_error",
    title="Baseline model - % error by price band",
    labels={
        "price_band": "Actual price band",
        "pct_error": "(actual - predicted) / actual × 100%",
    },
)
fig.update_yaxes(range=[-100, 100])
fig.add_hline(y=0, line_dash="dot")
fig.show()

# Median signed error per band: positive means the model under-predicts.
diag.groupby("price_band", observed=True).agg(
    n=("pct_error", "size"), median_pct_error=("pct_error", "median")
).round(1)

,n,median_pct_error
price_band,,
<$200K,707,-8.8
$200-350K,1911,-0.3
$350-500K,1972,0.8
$500-750K,1380,2.4
$750K-1M,629,4.9
>$1M,456,13.0


:::{caution} A common failure mode

The table shows that the model under-predicts the most expensive homes (a
median error of about +13% above \$1M) and over-predicts the cheapest ones
(about -9% below \$200K). Tree-based models pull predictions toward the middle
of the price distribution. QCP does not buy trophy homes, but it does buy at
the low end, where this bias would make cheap homes look more valuable than
they are. Always check the slice of the data you actually care about.

:::

## 📊 Predicted vs. Actual

The classic AVM diagnostic chart. Points should hug the diagonal.


In [13]:
plot_sample = diag.sample(n=min(3000, len(diag)), random_state=0)

fig = px.scatter(
    plot_sample,
    x="actual",
    y="predicted",
    color="market",
    opacity=0.5,
    title="Baseline model - predicted vs. actual last sold price",
    labels={
        "actual": "Actual last sold price (USD)",
        "predicted": "Predicted last sold price (USD)",
    },
)
lim = max(plot_sample["actual"].max(), plot_sample["predicted"].max())
fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(dash="dash"))
fig.update_xaxes(range=[0, 1_200_000])
fig.update_yaxes(range=[0, 1_200_000])
fig.show()

## 🌟 Feature Importance - Which Inputs Matter?

XGBoost exposes a built-in `feature_importances_` attribute. Combined with
the one-hot-encoded names from the preprocessor, we can build a clean
ranking.


First, recover the post-encoding feature names from the pipeline.


In [14]:
ohe = (
    baseline_model.named_steps["preprocess"]
    .named_transformers_["cat"]
    .named_steps["ohe"]
)
cat_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names


Then plot the top-20 most important features.


In [15]:
importances = (
    pd.Series(
        baseline_model.named_steps["regressor"].feature_importances_,
        index=all_names,
    )
    .sort_values(ascending=False)
    .head(20)
)

imp_df = importances.sort_values().reset_index()
imp_df.columns = ["feature", "importance"]

fig = px.bar(
    imp_df,
    x="importance",
    y="feature",
    orientation="h",
    title="Top-20 features by XGBoost importance",
    labels={"importance": "Importance", "feature": ""},
    height=600,
)
fig.show()


## 💾 Save the Baseline Model & Predictions

Part 2b will need the same train/test split so the comparison is honest. We
save the row indices, not the model itself, because the model is small and
quick to retrain.


In [16]:
splits = pd.DataFrame(
    {
        "row_index": list(X_train.index) + list(X_test.index),
        "split": ["train"] * len(X_train) + ["test"] * len(X_test),
    }
)
splits.to_parquet(f"{DATA_DIR}/baseline_split.parquet", index=False)

baseline_metrics = {
    "MAE": float(mean_absolute_error(y_test_raw, baseline_pred_test)),
    "RMSE": float(np.sqrt(mean_squared_error(y_test_raw, baseline_pred_test))),
    "R2": float(r2_score(y_test_raw, baseline_pred_test)),
    "MAPE": float(
        np.mean(np.abs((y_test_raw - baseline_pred_test) / y_test_raw)) * 100
    ),
}
print("Baseline test metrics:", baseline_metrics)


Baseline test metrics: {'MAE': 73863.609375, 'RMSE': 148555.98055951836, 'R2': 0.8020185828208923, 'MAPE': 17.080080682419407}


## 🧠 Investment-Committee Takeaways (Part 2a)

What the baseline shows, based on the outputs above:

- The baseline AVM predicts the most recent sold price with a MAPE of about
  17% on the held-out test set (MAE about \$74K, R² about 0.80). The median
  error per market is lower, about 7% to 12%.
- Errors are largest in Ann Arbor, Urbana, Charlottesville, and Champaign
  (median error about 12%) and at the price extremes: the model under-predicts
  homes above \$1M and over-predicts homes below \$200K.
- Location dominates the importance ranking (campus, state, and city
  indicators fill most of the top 10), followed by `living_area` and `baths`.
  One-hot encoding splits a location's importance across many columns, so
  read the ranking as a rough guide.

In Part 2b we test whether the listing description (free text) can push that
error lower.